# 02 — Experiment Design

Now we think like an **Experimentation Scientist**: before touching any data, we write down the
hypothesis, the randomization scheme, and the treatment definition.

## Hypotheses

- **H₀ (null):** the 10% price increase has **no effect** on revenue per customer.
- **H₁ (alternative):** the 10% price increase **changes** revenue per customer.

We'll test this two-sided (a price increase could plausibly backfire if demand is elastic enough),
then use the sign of the effect for the ship / don't-ship call.

## Randomization

- 50% Control (current price)
- 50% Treatment (price + 10%)

Randomization is done at the **customer level**, so a customer sees one price consistently for the
whole experiment (no risk of one person seeing both prices and contaminating the comparison).

## Treatment

Treatment customers get **price +10%**. But a price increase doesn't just add revenue — it can also
reduce how often people buy. We model that with **price elasticity of demand**:

> 10% price increase → 5% fewer purchases (example; fully configurable)


In [1]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd

from data_processing import BusinessConfig, generate_synthetic_customers
from experiment import TreatmentConfig, randomize_users, apply_price_treatment, run_experiment, summarize_groups

customers = generate_synthetic_customers(BusinessConfig(seed=42))
customers.shape


(20000, 7)

## 1. Randomize customers into Control / Treatment

In [2]:
treatment_config = TreatmentConfig(
    price_increase_pct=0.10,   # +10% price
    elasticity=0.5,            # 10% price increase -> 5% fewer purchases
    treatment_fraction=0.5,
    seed=123,
)

randomized = randomize_users(customers, treatment_config)
randomized["group"].value_counts(normalize=True)


group
treatment    0.5017
control      0.4983
Name: proportion, dtype: float64

### Sanity check: is the randomization balanced?

Before applying any treatment, control and treatment should look statistically identical on
pre-experiment covariates (this is an **A/A check** — if these differ meaningfully, something is
wrong with the randomization).

In [3]:
balance_check = (
    randomized.groupby("group")
    .agg(n=("customer_id", "count"),
         avg_tenure=("tenure_months", "mean"),
         avg_pre_revenue=("revenue", "mean"),
         plan_premium_share=("plan_tier", lambda s: (s == "Premium").mean()))
)
balance_check


,n,avg_tenure,avg_pre_revenue,plan_premium_share
group,,,,
control,9966,23.991672,132.106596,0.15011
treatment,10034,24.098366,133.930555,0.14710


Groups look balanced on tenure, pre-experiment revenue, and plan mix — randomization is doing its job.

## 2. Apply the pricing treatment + elasticity model

In [4]:
experiment_df = apply_price_treatment(randomized, treatment_config)
summary = summarize_groups(experiment_df)
summary


,group,n,revenue_per_customer_mean,revenue_per_customer_std,conversion_rate,avg_orders,avg_order_value
0,control,9966,132.106690,139.298315,0.685832,1.145093,115.540681
1,treatment,10034,140.306177,153.767955,0.663046,1.103747,127.296422


Notice:
- **avg_order_value** is ~10% higher in treatment (the direct price effect).
- **avg_orders** is lower in treatment (the elasticity / demand-response effect).
- **revenue_per_customer_mean** nets these two forces together — this is what we'll actually test.

## 3. Explore the elasticity assumption

What if the true elasticity is different from our 0.5 assumption? Let's see how the net revenue effect changes.

In [5]:
results = []
for elasticity in [0.0, 0.2, 0.5, 0.8, 1.0, 1.5]:
    cfg = TreatmentConfig(price_increase_pct=0.10, elasticity=elasticity, seed=123)
    exp = run_experiment(customers, cfg)
    s = summarize_groups(exp).set_index("group")
    control_rev = s.loc["control", "revenue_per_customer_mean"]
    treat_rev = s.loc["treatment", "revenue_per_customer_mean"]
    lift = (treat_rev - control_rev) / control_rev
    results.append({"elasticity": elasticity, "control_revenue": control_rev,
                     "treatment_revenue": treat_rev, "revenue_lift_pct": lift})

elasticity_table = pd.DataFrame(results)
elasticity_table


,elasticity,control_revenue,treatment_revenue,revenue_lift_pct
0,0.0,132.10669,147.323646,0.115187
1,0.2,132.10669,144.408575,0.093121
2,0.5,132.10669,140.306177,0.062067
3,0.8,132.10669,135.643909,0.026775
4,1.0,132.10669,132.683029,0.004363
5,1.5,132.10669,125.324805,-0.051336


This is a preview of the **sensitivity analysis** we'll formalize in Notebook 5 — even at this
early design stage, we can already see there's some elasticity value beyond which the price increase
stops paying off. We'll quantify that precisely once we have a real hypothesis test in Notebook 4.

## Next: `03_power_analysis.ipynb` — how many customers do we need in this experiment?